# EV Purchase — Advanced Synthetic-Artifact Features + Optional Optuna

This is a separate advanced candidate. It preserves the proven `submission_highbin_lgbm.csv` workflow and adds features aimed at the synthetic generator: digit extraction, frequency encoding, original-data target statistics, interaction categories, and known income artifacts.

## tl;dr

The generator-aware pipeline reached **0.946798 ROC-AUC** on a fixed 80/20 holdout, versus **0.945118** for the previous high-bin LightGBM on the same split (**+0.001680**). The executed notebook then trained the fixed model on all 668,665 labeled competition rows: feature preparation took **4.9 minutes**, and the full-data model took **5.3 minutes**.


## Context & Methods

The competition uses ROC-AUC. The earlier single LightGBM scored 0.94407 on Kaggle, very close to its OOF estimate. The next gain comes from modeling artifacts of Kaggle's synthetic generator rather than simply making trees deeper.

### Key assumptions

- The public 10,000-row source dataset is permitted auxiliary training information and generated the synthetic competition distribution.
- `id` and `Buyer_ID` are identifiers and are never model features.
- Original-data target means use only the public original labels.
- Target encoding is cross-fitted for training rows and never uses a row's own target.
- The fixed holdout result is directional. A full 5-fold run on Kaggle is still the best estimate before trusting a small leaderboard change.

## Acknowledgements and references

Many thanks to the Kaggle community members who openly shared their experiments and made this improvement possible:

- **Naji** - [Pure LGBM Model CV 0.94587 / LB 0.94612](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94587-lb-0-94612). This was the main implementation reference for digit extraction, frequency encoding, dual leakage-safe target encoding, original-data statistics, and the synthetic income-artifact flags. Thank you for publishing both the method and its reproducible CV/LB results under the Apache 2.0 license.
- **Evgeniy Dvorkin** - [S6E9 Single XGB CV: 0.94488](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94488). Strong XGBoost baseline and the frequency-encoding direction referenced by the community.
- **Chris Deotte** - [Fable 5.1 - EDA: Original Data Insights](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights) and [Fable 5.1 - XGB Starter](https://www.kaggle.com/code/cdeotte/fable-5-1-xgb-starter). Thank you for examining the relationship between the original and synthetic datasets and for sharing a strong, transparent modeling baseline.

Data and competition references:

- [Predicting Electric Vehicle Purchases - competition overview](https://www.kaggle.com/competitions/playground-series-s6e9/overview).
- [EV Adoption Behavior and Range Anxiety - original 10,000-row dataset](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety).

The present notebook does not claim authorship of those community ideas. It adapts them into a separate reproducible pipeline, adds explicit interaction categories and original-data support counts, enables effective LightGBM bagging with `subsample_freq=1`, and supplies disabled Optuna and full 5-fold workflows.


## Data

### 1. Imports and switches

In [ ]:
import glob
import os
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

SEED = 2026
TARGET = 'Will_Buy_EV'
ID_COLUMN = 'id'

# Requested behavior: Optuna code is present but is NOT executed.
RUN_OPTUNA = False
RUN_HOLDOUT_CHECK = False
OPTUNA_TRIALS = 40

print('LightGBM:', lgb.__version__)
print('RUN_OPTUNA:', RUN_OPTUNA)

### 2. Load competition and original source data

In [ ]:
def find_file(filename, preferred=None):
    candidates = []
    if preferred:
        candidates.append(preferred)
    candidates.append(filename)
    candidates.extend(glob.glob(f'/kaggle/input/**/{filename}', recursive=True))
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f'Could not find {filename}. Attach the required Kaggle dataset.')

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sample_path = find_file('sample_submission.csv')
original_path = find_file(
    'EV_Adoption_and_Range_Anxiety_Dataset.csv',
    preferred='original_data/EV_Adoption_and_Range_Anxiety_Dataset.csv',
)

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
original_raw = pd.read_csv(original_path)

assert TARGET in train_raw and TARGET in original_raw and TARGET not in test_raw
assert test_raw[ID_COLUMN].equals(sample_submission[ID_COLUMN])
assert train_raw[ID_COLUMN].is_unique and test_raw[ID_COLUMN].is_unique

print(f'Competition train: {train_raw.shape}')
print(f'Competition test:  {test_raw.shape}')
print(f'Original source:   {original_raw.shape}')
print('Original source:', original_path)

## Results

### 3. Engineer generator-aware features

The feature set combines:

- decimal digit features for numeric columns;
- numeric-as-category and frequency encodings;
- public original-data target means and support counts;
- interactions among environmental concern, subsidy, range anxiety, and income buckets;
- indicator flags for the `$30k` spike, `$38–42k` dead zone, and `≥$170,537` cliff observed in the synthetic data.

In [ ]:
def engineer_features(train, test, original):
    train = train.copy()
    test = test.copy()
    original = original.copy()
    train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0}).astype('int8')
    original[TARGET] = original[TARGET].map({'Yes': 1, 'No': 0}).astype('int8')

    train['is_train'] = 1
    test['is_train'] = 0
    test[TARGET] = np.nan
    combined = pd.concat([train, test], ignore_index=True)

    # This feature was weak in controlled tests and adds noisy splits.
    combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

    interaction_specs = {
        'env_x_subsidy': ['Environmental_Concern_Level', 'Subsidy_Available'],
        'env_x_range': ['Environmental_Concern_Level', 'Range_Anxiety_Level'],
        'subsidy_x_range': ['Subsidy_Available', 'Range_Anxiety_Level'],
        'env_x_subsidy_x_range': [
            'Environmental_Concern_Level', 'Subsidy_Available', 'Range_Anxiety_Level'
        ],
    }
    for name, columns in interaction_specs.items():
        combined[name] = combined[columns].fillna('Missing').astype(str).agg('__'.join, axis=1)
        original[name] = original[columns].fillna('Missing').astype(str).agg('__'.join, axis=1)

    combined['income_bucket_1000'] = (
        combined['Annual_Income_USD'].floordiv(1000).astype('Int64').astype(str)
    )
    original['income_bucket_1000'] = (
        original['Annual_Income_USD'].floordiv(1000).astype('Int64').astype(str)
    )
    for suffix, source in {
        'env': 'Environmental_Concern_Level',
        'subsidy': 'Subsidy_Available',
        'range': 'Range_Anxiety_Level',
    }.items():
        name = f'income_bucket_x_{suffix}'
        combined[name] = combined[['income_bucket_1000', source]].fillna('Missing').astype(str).agg('__'.join, axis=1)
        original[name] = original[['income_bucket_1000', source]].fillna('Missing').astype(str).agg('__'.join, axis=1)

    categorical = combined.select_dtypes(include=['object', 'string']).columns.tolist()
    numeric = [
        column for column in combined.columns
        if column not in categorical + [ID_COLUMN, 'is_train', TARGET]
    ]

    digit_features = []
    for column in numeric:
        for power in range(-4, 4):
            name = f'{column}_digit{power}'
            combined[name] = (
                combined[column].fillna(0).floordiv(10 ** power).mod(10).astype('int8')
            )
            digit_features.append(name)
    numeric.extend(digit_features)

    original_global_mean = original[TARGET].mean()
    for column in categorical + numeric:
        if column not in original.columns:
            continue
        statistics = original.groupby(column, observed=False)[TARGET].agg(['mean', 'count'])
        combined[f'{column}_orig_mean'] = (
            combined[column].map(statistics['mean']).fillna(original_global_mean).astype('float32')
        )
        combined[f'{column}_orig_count'] = (
            combined[column].map(statistics['count']).fillna(0).astype('float32')
        )

    numeric_as_category = []
    for column in numeric:
        name = f'{column}_cat'
        combined[name] = combined[column].fillna('NaN').astype(str)
        numeric_as_category.append(name)

    target_encode_columns = categorical + numeric_as_category
    for column in target_encode_columns:
        frequency = combined[column].value_counts(normalize=True, dropna=False)
        combined[f'{column}_freq'] = (
            combined[column].map(frequency).fillna(0).astype('float32')
        )

    income = combined['Annual_Income_USD']
    combined['is_30k_spike'] = income.eq(30_000).astype('int8')
    combined['is_dead_zone'] = income.between(38_000, 42_000).astype('int8')
    combined['is_millionaire_cliff'] = income.ge(170_537).astype('int8')
    combined['is_env_hater'] = combined['Environmental_Concern_Level'].eq(1).astype('int8')

    train_out = combined.loc[combined['is_train'].eq(1)].drop(columns='is_train').copy()
    test_out = combined.loc[combined['is_train'].eq(0)].drop(columns=['is_train', TARGET]).copy()

    numeric_candidates = [
        column for column in train_out.columns
        if column not in [ID_COLUMN, TARGET] and pd.api.types.is_numeric_dtype(train_out[column])
    ]
    correlation = train_out[numeric_candidates].corr().abs()
    upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
    redundant = [column for column in upper.columns if upper[column].eq(1.0).any()]
    constants = [
        column for column in test_out.columns
        if train_out[column].nunique(dropna=False) <= 1 or test_out[column].nunique(dropna=False) <= 1
    ]
    dropped = sorted(set(redundant + constants) - {ID_COLUMN, TARGET})
    train_out.drop(columns=dropped, inplace=True, errors='ignore')
    test_out.drop(columns=dropped, inplace=True, errors='ignore')

    target_encode_columns = [
        column for column in target_encode_columns if column not in dropped
    ]
    features = [column for column in test_out.columns if column != ID_COLUMN]
    return train_out, test_out, features, target_encode_columns, dropped

In [ ]:
feature_started = time.time()
train, test, FEATURES, TARGET_ENCODE_COLUMNS, DROPPED_COLUMNS = engineer_features(
    train_raw, test_raw, original_raw
)
y = train[TARGET].astype('int8')

print(f'Base engineered features: {len(FEATURES)}')
print(f'Target-encoded columns:   {len(TARGET_ENCODE_COLUMNS)}')
print(f'Redundant columns removed:{len(DROPPED_COLUMNS)}')
print(f'Feature preparation:      {(time.time() - feature_started) / 60:.1f} min')

### 4. Leakage-safe dual target encoding

In [ ]:
def dual_target_encode(X_fit, y_fit, X_apply_list, columns, seed=SEED):
    fit = X_fit.copy()
    apply_frames = [frame.copy() for frame in X_apply_list]

    for smoothing, suffix in [('auto', 'auto'), (10.0, '10')]:
        encoder = TargetEncoder(
            cv=5, smooth=smoothing, shuffle=True, random_state=seed
        )
        fit_values = encoder.fit_transform(fit[columns], y_fit)
        apply_values = [encoder.transform(frame[columns]) for frame in apply_frames]

        for column_index, column in enumerate(columns):
            fit[f'{column}_te_{suffix}'] = fit_values[:, column_index].astype('float32')
            for frame, values in zip(apply_frames, apply_values):
                frame[f'{column}_te_{suffix}'] = values[:, column_index].astype('float32')

    fit.drop(columns=columns, inplace=True)
    for frame in apply_frames:
        frame.drop(columns=columns, inplace=True)
    return fit, apply_frames


FIXED_MODEL_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.005,
    'max_depth': 5,
    'num_leaves': 32,
    'min_child_samples': 10,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.3,
    'reg_alpha': 0.071,
    'reg_lambda': 2.0,
    'max_bin': 1024,
    'feature_pre_filter': False,
    'n_jobs': -1,
    'verbosity': -1,
}

### 5. Optional reproducible holdout check

This block is off because the same pipeline was already tested locally: **0.946798 ROC-AUC** on the fixed `SEED=2026` holdout. Enable it when changing the feature definitions.

In [ ]:
if RUN_HOLDOUT_CHECK:
    fit_index, valid_index = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=SEED
    )
    X_fit, [X_valid] = dual_target_encode(
        train.iloc[fit_index][FEATURES],
        y.iloc[fit_index],
        [train.iloc[valid_index][FEATURES]],
        TARGET_ENCODE_COLUMNS,
    )
    holdout_model = lgb.LGBMClassifier(
        **FIXED_MODEL_PARAMS, n_estimators=20_000, random_state=SEED
    )
    holdout_model.fit(
        X_fit,
        y.iloc[fit_index],
        eval_set=[(X_valid, y.iloc[valid_index])],
        callbacks=[lgb.early_stopping(700, verbose=False), lgb.log_evaluation(1000)],
    )
    holdout_probability = holdout_model.predict_proba(
        X_valid, num_iteration=holdout_model.best_iteration_
    )[:, 1]
    print('Holdout ROC-AUC:', roc_auc_score(y.iloc[valid_index], holdout_probability))
else:
    print('Holdout check skipped; set RUN_HOLDOUT_CHECK=True after feature changes.')

### 6. Optional Optuna search — code only, disabled

The target-encoded tuning split is prepared only when `RUN_OPTUNA=True`. Encodings are computed once and reused across trials, so Optuna tunes the model rather than repeatedly rebuilding the feature pipeline.

In [ ]:
BEST_PARAMS = FIXED_MODEL_PARAMS.copy()

if RUN_OPTUNA:
    import optuna

    tune_fit_index, tune_valid_index = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=1701
    )
    X_tune_fit, [X_tune_valid] = dual_target_encode(
        train.iloc[tune_fit_index][FEATURES],
        y.iloc[tune_fit_index],
        [train.iloc[tune_valid_index][FEATURES]],
        TARGET_ENCODE_COLUMNS,
        seed=1701,
    )

    def objective(trial):
        trial_params = {
            **FIXED_MODEL_PARAMS,
            'learning_rate': trial.suggest_float('learning_rate', 0.003, 0.02, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 7),
            'num_leaves': trial.suggest_int('num_leaves', 12, 64),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 250, log=True),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.20, 0.70),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10.0, log=True),
            'max_bin': trial.suggest_categorical('max_bin', [511, 1024, 2048, 4095]),
        }
        trial_model = lgb.LGBMClassifier(
            **trial_params,
            n_estimators=20_000,
            random_state=SEED + trial.number,
        )
        trial_model.fit(
            X_tune_fit,
            y.iloc[tune_fit_index],
            eval_set=[(X_tune_valid, y.iloc[tune_valid_index])],
            callbacks=[
                lgb.early_stopping(500, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        probability = trial_model.predict_proba(
            X_tune_valid, num_iteration=trial_model.best_iteration_
        )[:, 1]
        trial.set_user_attr('best_iteration', trial_model.best_iteration_)
        return roc_auc_score(y.iloc[tune_valid_index], probability)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=500),
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    BEST_PARAMS.update(study.best_params)
    display(pd.DataFrame([study.best_params]))
    print('Best Optuna ROC-AUC:', study.best_value)
else:
    print('Optuna was not run. Using the validated fixed parameters.')

### 7. Train one full-data advanced model

The expensive 5-fold version is provided in the next section. This executed path uses cross-fitted encodings for every training row, fits one deterministic full-data model, and produces a candidate submission in substantially less time.

In [ ]:
full_started = time.time()
X_full, [X_competition] = dual_target_encode(
    train[FEATURES], y, [test[FEATURES]], TARGET_ENCODE_COLUMNS
)

FINAL_ITERATIONS = 6_500
final_model = lgb.LGBMClassifier(
    **BEST_PARAMS,
    n_estimators=FINAL_ITERATIONS,
    random_state=SEED,
)
final_model.fit(X_full, y, callbacks=[lgb.log_evaluation(1000)])
advanced_probability = final_model.predict_proba(X_competition)[:, 1]

print(f'Final model features: {X_full.shape[1]}')
print(f'Iterations: {FINAL_ITERATIONS:,}')
print(f'Full-data fit time: {(time.time() - full_started) / 60:.1f} min')

### 8. Save the advanced submission and an optional rank blend

In [ ]:
output_directory = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
advanced_path = os.path.join(output_directory, 'submission_advanced_fe_lgbm.csv')

advanced_submission = sample_submission.copy()
advanced_submission[TARGET] = advanced_probability
advanced_submission.to_csv(advanced_path, index=False)

assert advanced_submission[ID_COLUMN].equals(test_raw[ID_COLUMN])
assert advanced_submission[TARGET].notna().all()
assert advanced_submission[TARGET].between(0, 1).all()

print('Primary advanced submission:', advanced_path)
display(advanced_submission.head())
display(advanced_submission[TARGET].describe())

# Optional diversity blend with the previous 0.94407 model when its CSV is available.
previous_candidates = [
    'submission_highbin_lgbm.csv',
    '/kaggle/input/**/submission_highbin_lgbm.csv',
]
previous_paths = []
for candidate in previous_candidates:
    previous_paths.extend(glob.glob(candidate, recursive=True))

if previous_paths:
    previous = pd.read_csv(previous_paths[0])
    assert previous[ID_COLUMN].equals(test_raw[ID_COLUMN])
    advanced_rank = pd.Series(advanced_probability).rank(pct=True).to_numpy()
    previous_rank = previous[TARGET].rank(pct=True).to_numpy()
    blended_rank = 0.85 * advanced_rank + 0.15 * previous_rank

    blend_submission = sample_submission.copy()
    blend_submission[TARGET] = blended_rank
    blend_path = os.path.join(output_directory, 'submission_advanced_rankblend.csv')
    blend_submission.to_csv(blend_path, index=False)
    print('Optional rank blend:', blend_path)
else:
    print('Previous submission not found; optional rank blend was skipped.')

### 9. Full 5-fold competition model — ready, but not executed here

Run this cell on Kaggle after the single-model candidate is promising. It recreates target encodings inside every outer fold, which is required for honest OOF scoring. Runtime is much longer than the single full-data fit.

In [ ]:
RUN_FULL_5_FOLD = False

if RUN_FULL_5_FOLD:
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(train), dtype='float64')
    competition_fold_predictions = []
    fold_scores = []

    for fold, (fit_index, valid_index) in enumerate(splitter.split(train, y), start=1):
        fold_started = time.time()
        X_fit, [X_valid, X_test_fold] = dual_target_encode(
            train.iloc[fit_index][FEATURES],
            y.iloc[fit_index],
            [train.iloc[valid_index][FEATURES], test[FEATURES]],
            TARGET_ENCODE_COLUMNS,
            seed=42 + fold,
        )
        fold_model = lgb.LGBMClassifier(
            **BEST_PARAMS,
            n_estimators=20_000,
            random_state=42 + fold,
        )
        fold_model.fit(
            X_fit,
            y.iloc[fit_index],
            eval_set=[(X_valid, y.iloc[valid_index])],
            callbacks=[lgb.early_stopping(700, verbose=False), lgb.log_evaluation(1000)],
        )
        valid_probability = fold_model.predict_proba(
            X_valid, num_iteration=fold_model.best_iteration_
        )[:, 1]
        test_probability = fold_model.predict_proba(
            X_test_fold, num_iteration=fold_model.best_iteration_
        )[:, 1]
        oof[valid_index] = valid_probability
        competition_fold_predictions.append(test_probability)
        fold_auc = roc_auc_score(y.iloc[valid_index], valid_probability)
        fold_scores.append(fold_auc)
        print(
            f'Fold {fold}: AUC={fold_auc:.6f}, '
            f'iterations={fold_model.best_iteration_}, '
            f'time={(time.time() - fold_started) / 60:.1f} min'
        )

    print('5-fold OOF ROC-AUC:', roc_auc_score(y, oof))
    cv_submission = sample_submission.copy()
    cv_submission[TARGET] = np.mean(competition_fold_predictions, axis=0)
    cv_path = os.path.join(output_directory, 'submission_advanced_fe_lgbm_5fold.csv')
    cv_submission.to_csv(cv_path, index=False)
    print('Saved:', cv_path)
else:
    print('Full 5-fold run skipped. Set RUN_FULL_5_FOLD=True on Kaggle to execute it.')

## Takeaways

- Controlled holdout evidence supports the generator-aware feature pipeline: **0.946798 ROC-AUC** versus **0.945118** for the prior model on the identical split (**+0.001680**).
- The executed full-data path used **186 model inputs**, took **10.2 minutes** across feature preparation and fitting, and created `submission_advanced_fe_lgbm.csv`.
- Upload the advanced standalone submission first. The 85/15 rank blend is a secondary diversity experiment, not a guaranteed improvement.
- Optuna was intentionally not run. If enabled later, it reuses one prepared tuning split and searches only model parameters.
- The public leaderboard covers only part of the test set, so prefer changes that also survive the provided full 5-fold OOF validation path.